# Notebook 03 - Model Training

Trains four models (LR, RF, XGBoost, MLP), compares on Brier score and AUC-ROC,
and produces calibration curves.

Models are tracked in MLflow under the `nba-predictor` experiment.

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import mlflow
import pandas as pd

from src.models.train import (
    FEATURE_COLS,
    EXPERIMENT_NAME,
    load_and_split,
    train_all,
)
from src.models.evaluate import compute_metrics, plot_calibration_curves
from src.models.predict import load_best_pipeline, load_pipeline_by_run_id

## 1. Load Features

In [ ]:
FEATURES_PATH = "../data/processed/features.csv"

X_train, y_train, X_val, y_val, X_test, y_test = load_and_split(FEATURES_PATH)

print(f"Features: {len(FEATURE_COLS)} columns")
print(f"Train: {len(X_train):,} rows")
print(f"Val:   {len(X_val):,} rows")
print(f"Test:  {len(X_test):,} rows")

## 2. Split Summary by Season

In [ ]:
import pandas as pd

df_full = pd.read_csv(FEATURES_PATH, dtype={"GAME_ID": str})
df_full = df_full[df_full["is_bubble_game"] != 1]

from src.models.train import TRAIN_SEASONS, VAL_SEASONS, TEST_SEASONS

def label_split(season):
    if season in TRAIN_SEASONS:
        return "train"
    elif season in VAL_SEASONS:
        return "val"
    elif season in TEST_SEASONS:
        return "test"
    return "other"

df_full["split"] = df_full["SEASON"].apply(label_split)
split_summary = (
    df_full.groupby(["split", "SEASON"])
    .agg(n_games=("HOME_WIN", "count"), home_win_rate=("HOME_WIN", "mean"))
    .reset_index()
    .sort_values("SEASON")
)
split_summary

## 3. HOME_WIN Rate by Split

In [ ]:
for split_name, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
    rate = y.mean()
    n = len(y)
    print(f"{split_name:8s}: {rate:.3f} home win rate  (n={n:,})  baseline Brier = {rate*(1-rate):.4f}")

## 4. Train All Models

Set `RETRAIN = False` if you want to load from MLflow instead of retraining.

In [ ]:
RETRAIN = True  # Set to False to load existing runs from MLflow

if RETRAIN:
    run_ids = train_all(
        features_path=FEATURES_PATH,
        tune_xgb=False,  # Set True for Optuna tuning (slower)
    )
    print("\nRun IDs:")
    for name, rid in run_ids.items():
        print(f"  {name}: {rid}")
else:
    # Manually set run_ids from a previous MLflow run if needed
    run_ids = {}  # e.g., {"lr": "abc123", "rf": "def456", ...}

## 5. Model Comparison Table

In [ ]:
import numpy as np

model_name_map = {
    "lr":  "logistic_regression",
    "rf":  "random_forest",
    "xgb": "xgboost",
    "mlp": "mlp",
}

records = []
pipelines = {}

for short_name, run_id in run_ids.items():
    pipeline = load_pipeline_by_run_id(run_id)
    pipelines[short_name] = pipeline

    val_prob  = pipeline.predict_proba(X_val)[:, 1]
    test_prob = pipeline.predict_proba(X_test)[:, 1]

    val_m  = compute_metrics(y_val.values,  val_prob)
    test_m = compute_metrics(y_test.values, test_prob)

    records.append({
        "model":        model_name_map.get(short_name, short_name),
        "val_accuracy": val_m["accuracy"],
        "val_brier":    val_m["brier_score"],
        "val_auc":      val_m["roc_auc"],
        "test_brier":   test_m["brier_score"],
        "test_auc":     test_m["roc_auc"],
    })

comparison = (
    pd.DataFrame(records)
    .sort_values("val_brier")
    .reset_index(drop=True)
)
comparison.style.format({
    "val_accuracy": "{:.3f}",
    "val_brier":    "{:.4f}",
    "val_auc":      "{:.4f}",
    "test_brier":   "{:.4f}",
    "test_auc":     "{:.4f}",
}).highlight_min(subset=["val_brier", "test_brier"], color="lightgreen")

## 6. Calibration Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, (short_name, pipeline) in enumerate(pipelines.items()):
    val_prob = pipeline.predict_proba(X_val)[:, 1]
    full_name = model_name_map.get(short_name, short_name)
    plot_calibration_curves(
        {full_name: (y_val.values, val_prob)},
        ax=axes[i],
        title=f"Calibration - {full_name}",
    )

# Hide unused axes
for j in range(len(pipelines), 4):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig("../notebooks/calibration_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Model Selection Rationale

**Selection criterion: lowest validation Brier score.**

Brier score rewards well-calibrated probability estimates, which is the primary
goal of this project (calibrated home win probabilities, not just classifications).

XGBoost is expected to win on this tabular dataset. The MLP is included for
breadth but typically underperforms gradient boosting on structured data at this scale (~15k training rows).

Logistic Regression serves as the interpretable baseline with coefficient analysis.

In [ ]:
if records:
    best_row = comparison.iloc[0]
    print(f"Selected model: {best_row['model']}")
    print(f"  val_brier  = {best_row['val_brier']:.4f}")
    print(f"  test_brier = {best_row['test_brier']:.4f}")
    print(f"  val_auc    = {best_row['val_auc']:.4f}")
    
    spread = best_row['test_brier'] - best_row['val_brier']
    print(f"  val->test Brier spread: {spread:+.4f}")
    if abs(spread) > 0.01:
        print("  WARNING: spread > 0.01 - check for distribution shift")

## 8. MLflow UI

Run `mlflow ui` from the project root and visit http://localhost:5000 to inspect all runs.